### Analysing procedures per stage ###

We have two tables:
  * **ckd_enrollment**: Information about members
  * **ckd_claims**: Their claims (these include the combination of inpatient, outpatient and Rx claims)

The data spans the years 2017, 2018 and 2019.

In [1]:
# -----------------------------------------------------------------------------
# INITIALIZATION
# -----------------------------------------------------------------------------
import sys
print(f"Python version: {sys.version}")
import json
import logging
import csv
import gzip
import re
import pandas as pd
import numpy as np
from functools import reduce

import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession
from pyspark import SparkConf
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# -----------------------------------------------------------------------------
# INITIALIZE LOGGING
# -----------------------------------------------------------------------------
f = '%(asctime)-15s %(levelname)-8s %(message)s'
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")
logging.basicConfig(format=f)


from IPython.core.magic import register_cell_magic

# -----------------------------------------------------------------------------
# start_spark
# -----------------------------------------------------------------------------
def start_spark(
    driver_memory="100g",
    storage_fraction=0.5,
    num_nodes=10,
):
    """Initialize spark

    Arguments:
        driver_memory: Maximum heap size for the Spark driver Java
            virtual machine.
        storage_fraction: Controls what portion of Spark's unified
            memory is reserved for storage (i.e., caching/persisting data
            and broadcast variables), as a fraction of the total
            execution + storage memory pool.
            If you cache/persist a lot of data, and you're evicting
            data too early, you might increase this value (e.g. 0.6 or 0.7).
            Conversely, if your job is shuffle-heavy and fails due to
            memory pressure, you might decrease it (e.g. 0.3).
        num_nodes: How many concurrent threads to use while running
            in "local mode" (i.e. in a single machine instead of a cluster).
            Use '*' to use all cores, or an integer > 0 for a specific
            number of threads.
    """

    conf = SparkConf().setAppName("My_Application")
    conf.set("spark.driver.memory", driver_memory)
    conf.set("spark.memory.storageFraction", str(storage_fraction))
    conf.setMaster(f"local[{num_nodes}]")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel('WARN')

    return spark


Python version: 3.11.0 (main, Jun 13 2025, 14:48:45) [Clang 16.0.0 (clang-1600.0.26.6)]


In [2]:

spark = start_spark(num_nodes=10)
#spark.stop()

  
@register_cell_magic
def spark_sql(line, cell):
    result = spark.sql(cell)
    result.show(n=1000)
  

# -- READ ENROLLMENT AND DATA TABLES
enrollment_file = f"/Users/Charles/DATA/ckd/ckd_enrollment"
logger.info(f">>> Reading enrollment file: {enrollment_file}")
df_enrollment = spark.read.format("parquet").load(enrollment_file)
df_enrollment.createOrReplaceTempView('enrollment')
logger.info(f">>> ENROLLMENT has {df_enrollment.count():,} rows")
logger.info(f">>> ENROLLMENT has {df_enrollment.select('ENROLID').distinct().count():,} unique enrollees")

claims_file = f"/Users/Charles/DATA/ckd/ckd_claims"
logger.info(f">>> Reading claims file: {claims_file}")
df_claims = spark.read.format("parquet").load(claims_file)
df_claims.createOrReplaceTempView('claims')
logger.info(f">>> CLAIMS has {df_claims.count():,} rows")
logger.info(f">>> CLAIMS has {df_claims.select('ENROLID').distinct().count():,} unique enrollees")

# -- NOTE: with the "createOrReplaceTempView" we define a view of these
# -- tables, so we can use them in SQL queries.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/14 09:40:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/14 09:40:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2025-08-14 09:40:06,163 INFO     >>> Reading enrollment file: /Users/Charles/DATA/ckd/ckd_enrollment
25/08/14 09:40:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2025-08-14 09:40:07,746 INFO     >>> ENROLLMENT has 2,586,930 rows
2025-08-14 09:40:09,191 INFO     >>> ENROLLMENT has 862,310 unique enrollees    
2025-08-14 09:40:09,192 INFO     >>> Reading claims file: /Users/Charles/DATA/ckd/ckd_claims
2025-08-14 09:40

In [3]:
dx_cols = [col for col in df_claims.columns if col.__contains__("DX")]
# Step 2: Create CKD_STAGE using dynamic matching
def build_ckd_stage_case(dx_cols):
    stage_expr = None
    stage_map = {
        "N181": "CKD Stage 1",
        "N182": "CKD Stage 2",
        "N183": "CKD Stage 3",
        "N184": "CKD Stage 4",
        "N185": "CKD Stage 5",
        "N186": "ESRD",
        "N189": "Unspecified"
    }

    for code, label in stage_map.items():
        condition = reduce(lambda a, b: a | b, [F.col(c).startswith(code) for c in dx_cols])
        if stage_expr is None:
            stage_expr = F.when(condition, label)
        else:
            stage_expr = stage_expr.when(condition, label)
    if stage_expr is None:
        stage_expr = F.lit(None)
    return stage_expr

# Step 3: Filter for CKD codes and extract stage
ckd_filter = reduce(lambda a, b: a | b, [F.col(c).startswith("N18") for c in dx_cols])
ckd_stage_col = build_ckd_stage_case(dx_cols)
df_ckd_claims = (
    df_claims
    .filter(ckd_filter)
    .withColumn("CKD_STAGE", ckd_stage_col)
    .filter(F.col("CKD_STAGE").isNotNull())
    .filter(F.col("SVCDATE").isNotNull())
)


## Procedures


In [4]:
import json
with open("procgrp_map.json", "r") as f:
    procgrp_map = json.load(f)

# Build Spark map expression
from pyspark.sql.functions import create_map, lit

# Flattened list: [lit(k1), lit(v1), lit(k2), lit(v2), ...]
mapping_expr = []
for k, v in procgrp_map.items():
    mapping_expr.extend([lit(int(k)), lit(v)])

procgrp_map_expr = create_map(*mapping_expr)

# Filter: drop nulls AND exclude Dialysis code 140
df_procgrp_labeled = (
    df_ckd_claims
    .filter(F.col("PROCGRP").isNotNull() & (F.col("PROCGRP").cast("int") != 140))
    .withColumn("PROCGRP_Label", procgrp_map_expr.getItem(F.col("PROCGRP").cast("int")))
    .filter(F.col("PROCGRP_Label").isNotNull())  # drop unmapped
)

/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/pyspark/sql/classic/column.py:359: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


In [6]:
## First, get top 50 most common PROCGRP_Label values overall
top_50_labels = (
    df_procgrp_labeled
    .groupBy("PROCGRP_Label")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .limit(25)
    .select("PROCGRP_Label")
    .rdd.flatMap(lambda x: x)
    .collect()
)
# Filter to only top 50 PROCGRP_Label values
df_top_procgrp_stage = (
    df_procgrp_labeled
    .filter(F.col("PROCGRP_Label").isin(top_50_labels))
    .groupBy("PROCGRP_Label", "CKD_STAGE")
    .agg(F.count("*").alias("count"))
)


In [7]:
# Convert to pandas for plotting
df_top_procgrp_stage_pd = df_top_procgrp_stage.toPandas()

fig = px.bar(
    df_top_procgrp_stage_pd,
    x="PROCGRP_Label",
    y="count",
    color="CKD_STAGE",
    barmode="stack",
    title="Top 50 Procedure Groups by CKD Stage",
    labels={
        "PROCGRP_Label": "Procedure Group",
        "count": "Number of Claims",
        "CKD_STAGE": "CKD Stage"
    },
    height=700
)

fig.update_layout(xaxis_tickangle=45)
fig.show()


## Principal Diagnosis or DX1

within claims with CKD, 11613062 have NULL as PDX, 702669 PDX is not NULL! Hence I will retry the same analysis with DX1 feature instead where not NULL - NULL ratio is (12310661, 5070).


In [35]:
# Filter and get top 50 primary diagnosis codes
top_50_pdx = (
    df_ckd_claims
    .filter(F.col("DX1").isNotNull() & ~F.col("DX1").startswith("N18"))
    .groupBy("DX1")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .limit(20)
    .select("DX1")
    .rdd.flatMap(lambda x: x)
    .collect()
)

# Count per CKD stage
df_pdx_summary = (
    df_ckd_claims
    .filter(F.col("DX1").isin(top_50_pdx))
    .groupBy("DX1", "CKD_STAGE")
    .agg(F.count("*").alias("count"))
)

# Convert to pandas
df_pdx_summary_pd = df_pdx_summary.toPandas()

# Sort labels
pdx_order = (
    df_pdx_summary_pd.groupby("DX1")["count"]
    .sum()
    .sort_values(ascending=False)
    .index
)
print(df_pdx_summary_pd["DX1"].unique())

df_pdx_summary_pd["DX1"] = pd.Categorical(
    df_pdx_summary_pd["DX1"],
    categories=pdx_order,
    ordered=True
)



['I10' 'D509' 'D631' 'E1122' 'I120' 'Z940' 'E1165' 'R809' 'T82858A'
 'Z01818' 'N2581' 'D649' 'E119' 'I129' 'E43' 'E785' 'N179' 'Z0000' 'E559'
 'E782']


In [37]:
with open("icd_codes.json", "r") as f:
    icd_map = json.load(f)["icd10_label_map"]  # ✅ Fix is here

# Map ICD code to label only where it's in the top-50 summary
df_pdx_summary_pd["DX1"] = df_pdx_summary_pd["DX1"].astype(str)
df_pdx_summary_pd["DX1_Label"] = df_pdx_summary_pd["DX1"].map(icd_map).fillna("Unmapped")

fig = px.bar(
    df_pdx_summary_pd,
    x="DX1_Label",
    y="count",
    color="CKD_STAGE",
    barmode="stack",
    title="Top 20 First Diagnoses (DX1) by CKD Stage (excl. CKD diagnoses)",
    labels={"DX1_Label": "First Diagnosis", "count": "Number of Claims", "CKD_STAGE": "CKD Stage"},
    height=700
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

In [22]:
df_pdx = (
    df_ckd_claims
    .filter(F.col("DX1").isNotNull()))
df_nopdx = (
    df_ckd_claims
    .filter(F.col("DX1").isNull()))
df_pdx.count(),df_nopdx.count()

(12310661, 5070)

In [23]:
# Step 1: Filter to PDX + NETPAY is not null
df_pdx_cost = (
    df_ckd_claims
    .filter(F.col("DX1").isNotNull() & F.col("NETPAY").isNotNull())
    .groupBy("DX1")
    .agg(
        F.count("*").alias("n_claims"),
        F.countDistinct("ENROLID").alias("n_patients"),
        F.mean("NETPAY").alias("Avg_Cost_per_Claim"),
        F.sum("NETPAY").alias("Total_Cost")
    )
    .filter(F.col("n_claims") >= 100)  # ✅ Filter here for ≥100 claims
)

# Step 2: Convert to Pandas
df_pdx_cost_pd = df_pdx_cost.toPandas()

# Step 3: Map ICD code labels (assume icd_map = icd_codes_json["icd10_label_map"])
df_pdx_cost_pd["DX1"] = df_pdx_cost_pd["DX1"].astype(str)

# Get top 20 by average cost
df_top_cost = df_pdx_cost_pd.sort_values("Avg_Cost_per_Claim", ascending=False).head(20)


['Q675' 'A414' 'E1052' 'D595' 'M4316' 'K632' 'I7101' 'T84032A' 'D593'
 'E7521' 'T8642' 'M960' 'T82855A' 'T8132XA' 'I615' 'L88' 'M160' 'M48062'
 'T83518A' 'S37062A']


In [26]:

from pyspark.sql.functions import asc

with open("icd_codes.json", "r") as f:
    icd_map = json.load(f)["icd10_label_map"]  # ✅ Fix is here

print(df_top_cost["DX1"].unique())
df_top_cost["DX1_Label"] = df_top_cost["DX1"].map(icd_map).fillna("Unmapped")

fig = px.bar(
    df_top_cost,
    x="DX1_Label",
    y="Avg_Cost_per_Claim",
    color="n_patients",
    color_continuous_scale="Blues",
    title="Top 20 Most Expensive PDX Codes by Average Cost per Claim",
    labels={
        "PDX_Label": "Diagnosis",
        "Avg_Cost_per_Claim": "Average Cost per Claim ($)",
        "n_patients": "Number of Unique Patients"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    coloraxis_colorbar=dict(title="Patient Count")
)
#display(df_top_cost.sort_values(by="Avg_Cost_per_Claim",ascending=False))


['Q675' 'A414' 'E1052' 'D595' 'M4316' 'K632' 'I7101' 'T84032A' 'D593'
 'E7521' 'T8642' 'M960' 'T82855A' 'T8132XA' 'I615' 'L88' 'M160' 'M48062'
 'T83518A' 'S37062A']


# Analyse transplant patient trajectories

In [77]:
from pyspark.sql.functions import col, explode, array, when

# List of transplant codes
transplant_codes = ["Z940", "T8612", "T8619", "T8571XA", "T8611"]

# Select and stack all diagnosis columns into one column
df_diagnoses = (
    df_ckd_claims
    .select("ENROLID", array("PDX", "DX1", "DX2", "DX3", "DX4").alias("all_dx"))
    .withColumn("diag_code", explode("all_dx"))
    .filter(col("diag_code").isin(transplant_codes))
)

# Count the occurrences
df_transplant_counts = (
    df_diagnoses
    .groupBy("diag_code")
    .count()
    .orderBy("count", ascending=False)
)

df_transplant_counts.show()


+---------+------+
|diag_code| count|
+---------+------+
|     Z940|151214|
|    T8612| 16003|
|  T8571XA| 15549|
|    T8611| 14199|
|    T8619| 10023|
+---------+------+



In [ ]:
from pyspark.sql import functions as F

# Define codes of interest
transplant_related_codes = ["Z940", "T8612", "T8619", "T8571XA", "T8611"]

# Create a flag for any of the codes appearing in any diagnosis column
transplant_flag = (
    F.col("PDX").isin(transplant_related_codes) |
    F.col("DX1").isin(transplant_related_codes) |
    F.col("DX2").isin(transplant_related_codes) |
    F.col("DX3").isin(transplant_related_codes) |
    F.col("DX4").isin(transplant_related_codes)
)

# Filter for transplant-related CKD claims
df_transplant_claims = df_ckd_claims.filter(transplant_flag)

# Get number of unique CKD patients with transplant-related claims
n_transplant = df_transplant_claims.select("ENROLID").distinct().count()

# Total number of CKD patients
n_ckd = df_ckd_claims.select("ENROLID").distinct().count()

# Proportion
print(f"CKD patients with transplant-related claims: {n_transplant}")
print(f"Total CKD patients: {n_ckd}")
print(f"Proportion: {n_transplant / n_ckd:.2%}")


CKD patients with transplant-related claims: 7488

Total CKD patients: 191165

Proportion: 3.92%

In [62]:
transplant_codes = ["Z940", "T8612", "T8619", "T8571XA", "T8611"]

# Flag transplant claims
df_ckd_flagged = df_ckd_claims.withColumn(
    "has_transplant_code",
    F.col("PDX").isin(transplant_codes) |
    F.col("DX1").isin(transplant_codes) |
    F.col("DX2").isin(transplant_codes) |
    F.col("DX3").isin(transplant_codes) |
    F.col("DX4").isin(transplant_codes)
)
df_first_transplant = (
    df_ckd_flagged
    .filter("has_transplant_code")
    .groupBy("ENROLID")
    .agg(F.min("SVCDATE").alias("first_transplant_date"))
)
df_ckd_with_timing = (
    df_ckd_flagged.join(df_first_transplant, on="ENROLID", how="left")
    .withColumn("days_since_transplant",
        F.datediff(F.col("SVCDATE"), F.col("first_transplant_date"))
    )
)



In [83]:
df_ckd_with_txn = df_ckd_flagged.join(df_first_transplant, on="ENROLID", how="inner")
df_ckd_with_txn = df_ckd_with_txn.withColumn(
    "days_from_transplant", F.datediff("SVCDATE", "first_transplant_date")
).withColumn(
    "period", F.when(F.col("days_from_transplant") < 0, "Pre")
               .when(F.col("days_from_transplant") > 0, "Post")
               .otherwise("Unknown")
)
df_stage_trajectory = (
    df_ckd_with_txn.groupBy("period", "CKD_STAGE")
    .agg(F.countDistinct("ENROLID").alias("n_patients"))
    .orderBy("period", F.desc("n_patients"))
)


In [84]:
df_stage_trajectory_pd = df_stage_trajectory.toPandas()

import plotly.express as px
fig = px.bar(
    df_stage_trajectory_pd,
    x="CKD_STAGE",
    y="n_patients",
    color="period",
    barmode="group",
    title="CKD Stages Before and After Kidney Transplant",
    labels={"n_patients": "Number of Enrollees", "CKD_STAGE": "CKD Stage"}
)
fig.show()


In [73]:
from pyspark.sql import functions as F

# Patients who were ESRD before transplant
df_esrd_before_txn = (
    df_ckd_with_txn
    .filter((F.col("CKD_STAGE") == "ESRD") & (F.col("SVCDATE") <= F.col("first_transplant_date")))
    .select("ENROLID")
    .distinct()
)

# CKD claims after transplant for those patients
df_post_txn_stages = (
    df_ckd_with_txn
    .join(df_esrd_before_txn, on="ENROLID", how="inner")
    .filter(F.col("SVCDATE") > F.col("first_transplant_date"))
    .select("ENROLID", "CKD_STAGE")
    .distinct()
)


# Count post-transplant stages
df_stage_transition_counts = (
    df_post_txn_stages
    .groupBy("CKD_STAGE")
    .agg(F.countDistinct("ENROLID").alias("n_patients"))
    .orderBy(F.desc("n_patients"))
)

df_stage_transition_counts.show()


+-----------+----------+
|  CKD_STAGE|n_patients|
+-----------+----------+
|       ESRD|      3155|
|Unspecified|      1830|
|CKD Stage 3|      1110|
|CKD Stage 5|       962|
|CKD Stage 4|       767|
|CKD Stage 2|       439|
|CKD Stage 1|       126|
+-----------+----------+



Studies like Meier-Kriesche et al., 2004 (PubMed ID: 15280182) show median eGFR post-transplant is often in the Stage 2–3 range, depending on time post-transplant.

In [74]:
import plotly.graph_objects as go

# Convert to Pandas (after summarizing transitions in PySpark)
df_stage_transition_pd = df_stage_transition_counts.toPandas()

# Define source (from ESRD) and targets (post-transplant stages)
source_labels = ["ESRD (Pre-Transplant)"]
target_labels = df_stage_transition_pd["CKD_STAGE"].tolist()
values = df_stage_transition_pd["n_patients"].tolist()

# Merge all labels for indexing
all_labels = source_labels + target_labels

# Sankey requires numeric index for source/target
source_indices = [all_labels.index("ESRD (Pre-Transplant)")] * len(target_labels)
target_indices = [all_labels.index(stage) for stage in target_labels]

# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_labels,
        color="blue"
    ),
    link=dict(
        source=source_indices,
        target=target_indices,
        value=values
    )
)])

fig.update_layout(title_text="CKD Stage Flow from ESRD Post-Transplant", font_size=12)
fig.show()



In [ ]:
# Create list of ICD-10 codes to flag rejection/failure
rejection_codes = ["T8610", "T8611", "T8612", "T8619"]  # strip decimal for easier matching

# Define a helper column to flag rejection in any diagnosis field
df_with_rejection_flag = (
    df_ckd_with_txn.withColumn("has_rejection",
        F.expr("DX1 in ('{}') OR DX2 in ('{}') OR DX3 in ('{}') OR DX4 in ('{}') OR PDX in ('{}')"
            .format(*[','.join(rejection_codes)]*5))
    )
)
# Focus only on patients still marked as ESRD after transplant
df_esrd_post_txn = df_ckd_with_txn.filter(
    (F.col("CKD_STAGE") == "ESRD") & (F.col("SVCDATE") > F.col("first_transplant_date"))
)

# Count how many had rejection/failure codes
df_summary = (
    df_esrd_post_txn.withColumn("has_rejection", 
        F.col("PDX").isin(rejection_codes))  # or use expression as above
    .groupBy("has_rejection")
    .agg(F.countDistinct("ENROLID").alias("n_patients"))
)

df_summary.show()


+-------------+----------+
|has_rejection|n_patients|
+-------------+----------+
|         NULL|      3883|
|         true|       411|
|        false|      2187|
+-------------+----------+



In [ ]:
# Group by CKD stage and rejection status to count transitions
df_transition_labeled = (
    df_esrd_post_txn
    .groupBy("CKD_STAGE", "has_rejection")
    .agg(F.countDistinct("ENROLID").alias("n_patients"))
    .toPandas()
)

print(df_transition_labeled)
import plotly.graph_objects as go

# Build labels
source_labels = ["ESRD (Pre-Transplant)"]
outcome_labels = [
    f"{stage} - {'Rejection' if reject else 'Success'}"
    for stage, reject in zip(df_transition_labeled["CKD_STAGE"], df_transition_labeled["has_rejection"])
]
all_labels = source_labels + list(sorted(set(outcome_labels)))

# Map source and target indices
source_indices = [all_labels.index("ESRD (Pre-Transplant)")] * len(outcome_labels)
target_indices = [all_labels.index(label) for label in outcome_labels]

# Values
values = df_transition_labeled["n_patients"].tolist()

# Sankey plot
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_labels,
        color="blue"
    ),
    link=dict(
        source=source_indices,
        target=target_indices,
        value=values
    )
)])

fig.update_layout(
    title_text="CKD Stage Flow from ESRD Post-Transplant: Success vs Rejection",
    font_size=12
)
fig.show()
